### Parameter sweep for Model 1

In [1]:
cd ..

/home/veronika/Documents/2D-lymph-node-vertex-model


In [9]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path
import random
import traceback

import itertools
import os
import pickle

# Importing my modules

import src.vertexModel1 as vertexModel1
import src.inputMechanicalParametersModel1 as MechanicalParams1

import src.vertexModel2 as vertexModel2
import src.inputMechanicalParametersModel2 as MechanicalParams2



import src.auxFunctions as auxFunctions

# Set up plotting style
plt.style.use('seaborn-v0_8')
%matplotlib inline

In [19]:
def run_simulation_model1(num_ablations, cellmap, geom, energyContributions_model, combo_id, output_dir):
    """Run multiple ablations on the same tissue (Model 1 version, no line tension)."""
    
    all_recoil_k_pairs = []
    all_total_displacements = []

    for ablation_idx in range(num_ablations):
        print(f"Performing laser ablation {ablation_idx + 1}/{num_ablations}")
        
        ablation_map = cellmap.copy()
        
        # Select random interior edge
        # Find interior edges (not on boundary)
        boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = \
            auxFunctions.identify_boundary_layers(ablation_map, max_layers=6)
        
        chosen_edge = random.choice(inside_edges)
        
        srce, trgt = ablation_map.edge_df.loc[chosen_edge, ["srce", "trgt"]]
        
        # Find opposite edge
        opposite_edge = ablation_map.edge_df[
            (ablation_map.edge_df["srce"] == trgt) & 
            (ablation_map.edge_df["trgt"] == srce)
        ].index.tolist()[0]
        
        both_edges = [chosen_edge, opposite_edge]
        
        # Set elasticity to 0 (Model 1 only has length_elasticity, no line_tension)
        ablation_map.edge_df.loc[both_edges, 'length_elasticity'] = 0
        
        # Record initial distance
        initial_distance = np.linalg.norm(
            ablation_map.vert_df.loc[srce, ['x', 'y']].values - 
            ablation_map.vert_df.loc[trgt, ['x', 'y']].values
        )
        
        # Run post-ablation simulation
        energyContributions_model.compute_energy(ablation_map)
        ablation_map, geom, model_H, history_H, solver = vertexModel1.solveEuler(
            ablation_map, geom, energyContributions_model, endTime=50
        )
        
        # Track displacement
        displacement = []
        time_steps = []
        
        for t, cellmap_t in solver.history:
            current_distance = np.linalg.norm(
                cellmap_t.vert_df.loc[srce, ['x', 'y']].values - 
                cellmap_t.vert_df.loc[trgt, ['x', 'y']].values
            )
            displacement.append(current_distance - initial_distance)
            time_steps.append(t)
        
        # Fit recoil model
        def recoil_model(x, initialrecoil, K):
            return (initialrecoil / K) * (1 - np.exp(-K * x))
        
        params, _ = curve_fit(
            recoil_model, time_steps, displacement,
            p0=[0.00001, 3], bounds=(0, np.inf)
        )
        initialrecoil, K = params
        
        all_recoil_k_pairs.append((float(initialrecoil), float(K)))
        all_total_displacements.append(float(displacement[-1]))
        
        print(f"Recoil = {initialrecoil:.5f}, K = {K:.5f}, Total displacement = {displacement[-1]:.5f}")
    
    return all_recoil_k_pairs, all_total_displacements

In [20]:
def run_model1_sweep(output_base="model1_sweep_results", max_retries=3):
    
    output_base = Path(output_base)
    output_base.mkdir(parents=True, exist_ok=True)
    
    # Model 1 parameter ranges
    area_elasticities = np.linspace(10.0, 100.0, 20)
    length_elasticities = np.linspace(10.0, 200.0, 20)
    viscosities = np.linspace(400.0, 1500.0, 20)
    
    all_combos = list(itertools.product(area_elasticities, length_elasticities, viscosities))
    
    def format_combo(a, l, v):
        return f"A{int(a)}_L{int(l)}_V{int(v)}"
    
    results = []
    
    for idx, (area_el, length_el, visc) in enumerate(all_combos):
        combo_id = format_combo(area_el, length_el, visc)
        output_dir = output_base / combo_id
        output_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\n🔬 Running: {combo_id}")
        
        retries = 0
        while retries < max_retries:
            try:
                # Initialize
                cellmap, geom, energy_model = vertexModel1.initialize()
                cellmap = MechanicalParams1.update(cellmap)
                
                # Set parameters
                cellmap.face_df['area_elasticity'] = area_el
                cellmap.edge_df['length_elasticity'] = length_el
                cellmap.vert_df['viscosity'] = visc
                
                # Relax
                energy_model.compute_energy(cellmap)
                cellmap, geom, _, _, _ = vertexModel1.solveEuler(cellmap, geom, energy_model, endTime=100)
                
                # Run ablations using your existing run_simulation function
                num_ablations = 3
                ablation_results, total_displacements = run_simulation_model1(
                    num_ablations=num_ablations,
                    cellmap=cellmap.copy(),
                    geom=geom,
                    energyContributions_model=energy_model,
                    combo_id=combo_id,
                    output_dir=output_dir
                )
                
                # Save results
                result_dict = {
                    "combo_id": combo_id,
                    "area_elasticity": float(area_el),
                    "length_elasticity": float(length_el),
                    "viscosity": float(visc),
                    "ablations": [
                        {"initial_recoil": r[0], "K": r[1], "total_displacement": d}
                        for r, d in zip(ablation_results, total_displacements)
                    ]
                }
                
                with open(output_dir / "results.json", "w") as f:
                    json.dump(result_dict, f, indent=2, default=convert_numpy)
                
                results.append(result_dict)
                print(f"✅ Complete: {combo_id}")
                break
                
            except Exception as e:
                print(f"⚠️ Error (attempt {retries+1}/{max_retries}): {e}")
                retries += 1
    
    return results

In [21]:
# Model 1 parameter ranges
area_elasticities = np.linspace(10.0, 100.0, 20)      # 20 values from 10 to 100
length_elasticities = np.linspace(10.0, 200.0, 20)   # 20 values from 10 to 200
viscosities = np.linspace(400.0, 1500.0, 20)  

In [22]:
results = run_model1_sweep(output_base="model1_sweep_results_test", max_retries=3)




🔬 Running: A10_L10_V400
Topology changed!
Performing laser ablation 1/3
Recoil = 0.06145, K = 0.09565, Total displacement = 0.65749
Performing laser ablation 2/3
Recoil = 0.06318, K = 0.09743, Total displacement = 0.66384
Performing laser ablation 3/3
Recoil = 0.05991, K = 0.09488, Total displacement = 0.64631
✅ Complete: A10_L10_V400

🔬 Running: A10_L10_V457
Topology changed!
Performing laser ablation 1/3
Recoil = 0.05776, K = 0.08473, Total displacement = 0.69012
Performing laser ablation 2/3
Recoil = 0.05263, K = 0.07692, Total displacement = 0.68693
Performing laser ablation 3/3
Recoil = 0.05871, K = 0.08202, Total displacement = 0.72466
✅ Complete: A10_L10_V457

🔬 Running: A10_L10_V515
Topology changed!
Performing laser ablation 1/3
Recoil = 0.04839, K = 0.07660, Total displacement = 0.63512
Performing laser ablation 2/3
Recoil = 0.04904, K = 0.07647, Total displacement = 0.64408
Performing laser ablation 3/3
Recoil = 0.04933, K = 0.07188, Total displacement = 0.68631
✅ Complete:

KeyboardInterrupt: 

### Parameter sweep for Model 2

In [19]:
def convert_numpy(obj):
    if isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        raise TypeError(f"Object of type {obj.__class__.__name__} is not JSON serialisable")


In [20]:
def run_simulation(num_ablations, cellmap, geom, energyContributions_model, combo_id, output_dir):
    all_recoil_k_pairs = []
    all_total_displacements = []

    for ablation_idx in range(num_ablations):
        print(f"\n💥 Performing laser ablation {ablation_idx + 1}/{num_ablations}")
        retries = 0
        max_retries = 3

        while retries < max_retries:
            try:
                ablation_map = cellmap.copy()
                boundary_edges, boundary_faces, inside_edges, outside_edges, inside_faces, inside_vertices, outside_vertices = auxFunctions.identify_boundary_layers(ablation_map, max_layers=6)
                chosen_edge = random.choice(inside_edges)

                srce, trgt = ablation_map.edge_df.loc[chosen_edge, ["srce", "trgt"]]
                opposite_edge = ablation_map.edge_df[
                    (ablation_map.edge_df["srce"] == trgt) &
                    (ablation_map.edge_df["trgt"] == srce)
                ].index.tolist()[0]
                both_edges = [chosen_edge, opposite_edge]

                ablation_map.edge_df.loc[both_edges, 'length_elasticity'] = 0
                ablation_map.edge_df.loc[both_edges, 'line_tension'] = 0

                energyContributions_model.compute_energy(ablation_map)
                ablation_map, geom, model_H, history_H, solver1 = vertexModel2.solveEuler(ablation_map, geom, energyContributions_model, endTime=50)

                ablation_output_dir = output_dir / f"ablation_{ablation_idx + 1}"
                ablation_output_dir.mkdir(parents=True, exist_ok=True)

                for t, cellmap_t in solver1.history:
                    time_pkl_path = ablation_output_dir / f"time_{t:06.2f}.pkl"
                    with open(time_pkl_path, "wb") as f:
                        pickle.dump(cellmap_t, f)

                srce_vert = srce
                trgt_vert = trgt
                displacement = []
                time_steps = []
                initial_distance = None

                for t, cellmap_t in solver1.history:
                    coords_srce = cellmap_t.vert_df.loc[srce_vert, ['x', 'y']]
                    coords_trgt = cellmap_t.vert_df.loc[trgt_vert, ['x', 'y']]
                    distance = np.linalg.norm(coords_srce - coords_trgt)

                    if initial_distance is None:
                        initial_distance = distance

                    displacement.append(distance - initial_distance)
                    time_steps.append(t)

                def recoil_model(x, initialrecoil, K):
                    return (initialrecoil / K) * (1 - np.exp(-K * x))

                params, _ = curve_fit(
                    recoil_model, time_steps, displacement,
                    p0=[0.00001, 3], bounds=(0, np.inf)
                )
                initialrecoil, K = params

                all_recoil_k_pairs.append((float(initialrecoil), float(K)))
                all_total_displacements.append(float(displacement[-1]))

                print(f"✅ Recoil = {initialrecoil:.5f}, K = {K:.5f}, Total displacement = {displacement[-1]:.5f}")

                plt.figure()
                plt.plot(time_steps, displacement, marker='o', label='Displacement')
                plt.xlabel('Time')
                plt.ylabel('Extra displacement')
                plt.title(f'Recoil Curve - Ablation {ablation_idx + 1}')
                plt.legend()
                plt.grid(False)
                
                #Set consistent axes
                plt.xlim(0, 50)   # time range is 0 to 50 steps
                plt.ylim(0, 2)

                plt.tight_layout()
                plt.savefig(ablation_output_dir / "recoil_curve.png")
                plt.close()

                with open(ablation_output_dir / "metrics.json", "w") as f:
                    json.dump({
                        "initial_recoil": float(initialrecoil),
                        "K": float(K),
                        "total_displacement": float(displacement[-1])
                    }, f, indent=2)

                break

            except Exception as e:
                retries += 1
                print(f"⚠️  Ablation {ablation_idx + 1} failed on attempt {retries}: {e}")
                if retries >= max_retries:
                    print(f"❌ Skipping ablation {ablation_idx + 1} after {max_retries} retries.")

    return all_recoil_k_pairs, all_total_displacements


In [ ]:
area_elasticities = [1, 10, 20, 50, 100]
length_elasticities = [10] + list(range(50, 400, 50))
viscosities = np.linspace(10, 1500, 11)
line_tensions = np.round(np.logspace(np.log10(1), np.log10(1000), 18)).astype(int)

In [23]:
import traceback

# --- Setup ---
max_retries = 3
results = []
output_base = Path("parameter_sweep_results_test")

# --- Resume from here ---
start_after_combo = None

# --- Generate all parameter combinations ---
all_combos = list(itertools.product(area_elasticities, length_elasticities, viscosities, line_tensions))

def format_combo(a, l, v, t):
    return f"A{int(a)}_L{int(l)}_V{int(v)}_T{int(t)}"

# --- Determine starting point ---
start_index = 0
for i, (a, l, v, t) in enumerate(all_combos):
    if format_combo(a, l, v, t) == start_after_combo:
        start_index = i + 1
        break

print(f"🔁 Resuming from index {start_index}/{len(all_combos)}: {format_combo(*all_combos[start_index])}")

# --- Main loop ---
for i, (area_el, length_el, visc, tension) in enumerate(all_combos[start_index:]):
    combo_id = format_combo(area_el, length_el, visc, tension)
    output_dir = output_base / combo_id
    result_file = output_dir / f"results_{combo_id}.json"

    # Optional skip (uncomment if you still have older results locally)
    # if result_file.exists():
    #     print(f"\n⏭️  Skipping {combo_id}, already done.")
    #     continue

    print(f"\n🔬 Running parameter combo: {combo_id}")
    output_dir.mkdir(parents=True, exist_ok=True)

    retries = 0
    while retries < max_retries:
        try:
            # --- Step 1: Initialise ---
            cellmap, geom, energyContributions_model = vertexModel2.initialize(40)

            # --- Step 2: Apply parameters ---
            cellmap.face_df["area_elasticity"] = area_el
            cellmap.edge_df["length_elasticity"] = length_el
            cellmap.edge_df["prefered_length"] = cellmap.edge_df["length"].mean()
            cellmap.edge_df["line_tension"] = tension
            cellmap.face_df["prefered_area"] = cellmap.face_df["area"].mean()
            cellmap.vert_df["viscosity"] = visc

            # --- Step 3: Save initial state ---
            auxFunctions.save_simulation_state(cellmap, output_dir, f"init_{combo_id}.pkl")

            # --- Step 4: Relax the tissue ---
            energyContributions_model.compute_energy(cellmap)
            cellmap, geom, energyContributions_model, history, solver = vertexModel2.solveEuler(
                cellmap, geom, energyContributions_model, endTime=100
            )
            auxFunctions.save_simulation_state(cellmap, output_dir, f"relaxed_{combo_id}.pkl")

            # --- Optional: Visualise tissue (commented out to save space) ---
            fig, ax = auxFunctions.view(cellmap, geom)
            fig.savefig(output_dir / f"relaxed_{combo_id}_view.png", dpi=300)
            plt.close(fig)

            # --- Step 5: Perform laser ablations ---
            num_ablations = 3
            ablation_results, total_displacements = run_simulation(
                num_ablations=num_ablations,
                cellmap=cellmap.copy(),
                geom=geom,
                energyContributions_model=energyContributions_model,
                combo_id=combo_id,
                output_dir=output_dir
            )

            # --- Step 6: Save results ---
            result_dict = {
                "combo_id": combo_id,
                "area_elasticity": area_el,
                "length_elasticity": length_el,
                "viscosity": visc,
                "line_tension": tension,
                "ablations": [
                    {"initial_recoil": r[0], "K": r[1], "total_displacement": d}
                    for r, d in zip(ablation_results, total_displacements)
                ]
            }

            with open(result_file, "w") as f:
                json.dump(result_dict, f, indent=2, default=convert_numpy)

            results.append(result_dict)
            break  # Exit retry loop

        except Exception as e:
            print(f"⚠️  Error in {combo_id} (attempt {retries+1}/{max_retries}): {e}")
            traceback.print_exc()
            retries += 1

    if retries >= max_retries:
        print(f"❌ Skipping {combo_id} after {max_retries} failed attempts.")

🔁 Resuming from index 0/7200: A1_L10_V10_T1

🔬 Running parameter combo: A1_L10_V10_T1
Simulation state saved at parameter_sweep_results_test/A1_L10_V10_T1/init_A1_L10_V10_T1.pkl
Topology changed!
Simulation state saved at parameter_sweep_results_test/A1_L10_V10_T1/relaxed_A1_L10_V10_T1.pkl

💥 Performing laser ablation 1/3
✅ Recoil = 0.00001, K = 3.00000, Total displacement = 0.00000

💥 Performing laser ablation 2/3
✅ Recoil = 0.00000, K = 6.00000, Total displacement = 0.00000

💥 Performing laser ablation 3/3
✅ Recoil = 0.03307, K = 17.84142, Total displacement = -0.02158

🔬 Running parameter combo: A1_L10_V10_T2
Simulation state saved at parameter_sweep_results_test/A1_L10_V10_T2/init_A1_L10_V10_T2.pkl
Topology changed!
Simulation state saved at parameter_sweep_results_test/A1_L10_V10_T2/relaxed_A1_L10_V10_T2.pkl

💥 Performing laser ablation 1/3
✅ Recoil = 0.12036, K = 2.56954, Total displacement = 0.04684

💥 Performing laser ablation 2/3
✅ Recoil = 0.14217, K = 1.69602, Total displace

KeyboardInterrupt: 